In [34]:
import numpy as np
from LiouvilleLanczos.Quantum_computer.Hamiltonian import Line_Hubbard, BoundaryCondition
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit.quantum_info import SparsePauliOp, Operator
from pauliarray.pauli.weighted_pauli_array import WeightedPauliArray
from pauliarray.pauli.pauli_array import PauliArray
from pauliarray.binary import symplectic
import pauliarray.pauli.pauli_array as p
from pauliarray.binary import bit_operations as bitops
from collections import defaultdict

mapper = JordanWignerMapper()

def fermionic_op_to_matrix(op):
    qubit_op = mapper.map(op)
    return np.asarray(qubit_op.to_matrix(), dtype=complex)

def to_sparse_pauli(op):
    """
    Convert FermionicOp -> SparsePauliOp.
    Leave SparsePauliOp unchanged.
    """
    if isinstance(op, SparsePauliOp):
        return op

    if isinstance(op, FermionicOp):
        return mapper.map(op).simplify(atol=1e-12)

    raise TypeError(f"Unsupported operator type: {type(op)}")

def to_pauli(op):
    labels = op.paulis.to_labels()
    weights = op.coeffs
    return WeightedPauliArray.from_labels_and_weights(labels=labels, weights=weights), PauliArray.from_labels(labels=labels)

def standard_symplectic_basis(n):
    Z = np.zeros((n, 2*n), dtype=bool)
    X = np.zeros((n, 2*n), dtype=bool)

    Z[np.arange(n), np.arange(n)] = True

    X[np.arange(n), n + np.arange(n)] = True
    return np.vstack([Z, X])

In [35]:
def pauli_commutator(P1,P2, duplicate = True):
    result = {}
    for A_name, A in P1.items():
        for B_name, B in P2.items():
            key = tuple(sorted([A_name, B_name]))
            if not duplicate and A_name == B_name:
                continue
            if key in result:
                continue
            A_paulis = A.paulis
            B_paulis = B.paulis
            if hasattr(A,"weights") and hasattr (B, "weights"):
                A_coeffs = np.asarray(A.weights, dtype=complex).reshape(-1)
                B_coeffs = np.asarray(B.weights, dtype=complex).reshape(-1)
            else:
                A_coeffs = np.ones(len(A_paulis.to_labels()))
                B_coeffs = np.ones(len(B_paulis.to_labels()))

            dic = defaultdict(complex)

            for i in range(len(A_paulis.to_labels())):
                a_val = A_paulis[i]
                Ca = A_coeffs[i]
                for j in range(len(B_paulis.to_labels())):
                    b_val = B_paulis[j]
                    Cb = B_coeffs[j]
                    comm, coeffs = p.commutator(a_val,b_val)
                    dic[comm.to_labels()[0]] += coeffs*Ca*Cb
                    
            dic = {
                label: coeff
                for label, coeff in dic.items()
                if abs(coeff) > 1e-12
            }
            result[tuple(sorted([A_name, B_name]))]=(len(dic) == 0)
    return result, dic

def pauli_anticommutator(P1,P2, duplicate = True):
    result = {}
    for A_name, A in P1.items():
        for B_name, B in P2.items():
            key = tuple(sorted([A_name, B_name]))
            if not duplicate and A_name == B_name:
                continue
            if key in result:
                continue
            A_paulis = A.paulis
            B_paulis = B.paulis
            if hasattr(A,"weights") and hasattr (B, "weights"):
                A_coeffs = np.asarray(A.weights, dtype=complex).reshape(-1)
                B_coeffs = np.asarray(B.weights, dtype=complex).reshape(-1)
            else:
                A_coeffs = np.ones(len(A_paulis.to_labels()))
                B_coeffs = np.ones(len(B_paulis.to_labels()))

            dic = defaultdict(complex)

            for i in range(len(A_paulis.to_labels())):
                a_val = A_paulis[i]
                Ca = A_coeffs[i]
                for j in range(len(B_paulis.to_labels())):
                    b_val = B_paulis[j]
                    Cb = B_coeffs[j]
                    comm, coeffs = p.anticommutator(a_val,b_val)
                    dic[comm.to_labels()[0]] += coeffs*Ca*Cb
            dic = {
                label: coeff
                for label, coeff in dic.items()
                if abs(coeff) > 1e-12
            }
            result[tuple(sorted([A_name, B_name]))]=(len(dic) == 0)
    return result, dic



In [36]:
U = 4 
Ham = Line_Hubbard(-1,U/2,U,3,boundary_condition=BoundaryCondition.OPEN)

weight_H, H = to_pauli(to_sparse_pauli(Ham))

In [37]:
zx_H = [pauli.zx_strings for pauli in H]
zx_H = np.array(zx_H)

zx_H = np.vstack(zx_H).astype(bool)     #stack zx strings to make one big zx-matrix instead of individual zx strings


In [161]:
H_ortho = symplectic.gram_schmidt_orthogonalization(zx_H)
centralizer = symplectic.coisotropic_subspace(H_ortho)
centralizer = symplectic.gram_schmidt_orthogonalization(centralizer)
sym = symplectic.isotropic_subspace(centralizer)
sym = np.array(symplectic.gram_schmidt_orthogonalization(sym))
sym_conj = symplectic.conjugate_subspace(sym)

B = np.vstack([sym,sym_conj])

n = B.shape[1] // 2
target = standard_symplectic_basis(n)


In [162]:
def symplectic_J(n):
    I = np.eye(n, dtype=bool)
    O = np.zeros((n, n), dtype=bool)
    return np.block([
        [O, I],
        [I, O]
    ])


n = sym.shape[1] // 2
J = symplectic_J(n).astype(int)


In [163]:
print("sym J sym.T:")
print((sym @ J @ sym.T % 2).astype(int))

print("sym_conj J sym_conj.T:")
print((sym_conj @ J @ sym_conj.T % 2).astype(int))

print("sym J sym_conj.T:")
print((sym @ J @ sym_conj.T % 2).astype(int))

sym J sym.T:
[[0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]]
sym_conj J sym_conj.T:
[[0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]
 [0 0 0 0 0 0]]
sym J sym_conj.T:
[[1 0 0 0 0 0]
 [0 1 0 0 0 0]
 [0 0 1 0 0 0]
 [0 0 0 1 0 0]
 [0 0 0 0 1 0]
 [0 0 0 0 0 1]]


In [164]:
def zx_to_pauli(zx):
    """
    Convert one ZX binary vector into a Pauli string.

    Convention:
        zx = (z0, z1, ..., z_{n-1} | x0, x1, ..., x_{n-1})

    Mapping per qubit:
        z=0, x=0 -> I
        z=0, x=1 -> X
        z=1, x=0 -> Z
        z=1, x=1 -> Y

    Returns:
        str, e.g. "IXYZ"
    """
    zx = np.asarray(zx, dtype=np.uint8) % 2

    if zx.ndim != 1:
        raise ValueError("zx_to_pauli expects one 1D ZX vector.")

    if len(zx) % 2 != 0:
        raise ValueError("ZX vector length must be even.")

    n = len(zx) // 2
    z = zx[:n]
    x = zx[n:]

    pauli = []

    for zi, xi in zip(z, x):
        if zi == 0 and xi == 0:
            pauli.append("I")
        elif zi == 0 and xi == 1:
            pauli.append("X")
        elif zi == 1 and xi == 0:
            pauli.append("Z")
        elif zi == 1 and xi == 1:
            pauli.append("Y")

    return "".join(pauli)

def zx_matrix_to_paulis(zx_matrix):
    zx_matrix = np.asarray(zx_matrix, dtype=np.uint8) % 2

    if zx_matrix.ndim != 2:
        raise ValueError("Expected a 2D ZX matrix.")

    return [zx_to_pauli(row) for row in zx_matrix]

In [178]:
F = J @ B.T @ J % 2


sym_new = sym @ F % 2
sym_conj_new = sym_conj @ F % 2

print(sym_new.shape)

(6, 12)


In [170]:
print(sym_new)

print(sym_conj_new)

[[1 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0]]
[[0 0 0 0 0 0 1 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 0 0]
 [0 0 0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 0 1]]


In [186]:
CHC_zx = H_ortho @ F % 2
CHC_zx = symplectic.gram_schmidt_orthogonalization(CHC_zx)

print(zx_matrix_to_paulis(CHC_zx))

['IIIZII', 'IIZZII', 'IZZIII', 'ZZZIII', 'IIIIII', 'ZIZIZI', 'ZIZIZI', 'ZIZIZI']
